In [ ]:
import os
FIG_ROOT = os.environ.get('SCDIS_FIG', os.path.join(os.environ['SCDIS_ROOT'], 'figures'))

In [1]:
import os
import numpy as np
import scanpy as sc
import pandas as pd
from itertools import combinations
from scipy.stats import pearsonr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import gseapy as gp

In [2]:
import matplotlib
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.family'] = 'Liberation Sans'
matplotlib.rcParams['mathtext.default'] = 'regular'
matplotlib.rcParams['axes.unicode_minus'] = False

# Config

In [3]:
BASE     = f'{os.environ["SCDIS_ROOT"]}'
CTRL_PATH = f'adata_structured_control.h5ad'
STIM_PATH = f'adata_structured_stimulated.h5ad'
ORG_PATH  = f'{BASE}/Datasets/preprocessed_datasets/kang.h5ad'
OUT_DIR   = f'out'

N_SHARED   = 20         
GSEA_GENESET = 'MSigDB_Hallmark_2020'

KNOWN_ISGS = {'ISG15','IFI6','IFIT1','IFIT3','MX1','OAS1','STAT1','IRF7',
              'IFI44L','LY6E','ISG20','MX2','OAS3','RSAD2','IFI44','IFITM3',
              'IFI35','OASL','BST2','HERC5','USP18','CXCL10','GBP1','TRIM22'}
os.makedirs(OUT_DIR, exist_ok=True)

# Read and normalize

In [4]:
ctrl = sc.read_h5ad(CTRL_PATH)
stim = sc.read_h5ad(STIM_PATH)
org  = sc.read_h5ad(ORG_PATH)

print(ctrl.X.max(), stim.X.max())

gene_names = org.var_names.values.copy()
ctrl.var_names = gene_names
stim.var_names = gene_names

for ad in [ctrl, stim]:
    sc.pp.normalize_total(ad, target_sum=1e4)
    sc.pp.log1p(ad)

print(ctrl.X.max(), stim.X.max())

/home/SE/miniconda3/envs/trials/lib/python3.10/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/SE/miniconda3/envs/trials/lib/python3.10/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


1096.6399 800.6345
8.088914 7.82658


In [5]:
celltypes = sorted(ctrl.obs['cell_type'].unique())
levels    = sorted(ctrl.obs['level'].unique())
pairs     = list(combinations(celltypes, 2))

# Compute cumulative and marginal perturbation effect

In [6]:
cum_pert = {}
for ct in celltypes:
    cum_pert[ct] = {}
    for lv in levels:
        mask = (ctrl.obs['level'] == lv).values & (ctrl.obs['cell_type'] == ct).values
        if mask.sum() == 0:
            continue
        cum_pert[ct][lv] = np.array(
            (stim.X[mask] - ctrl.X[mask]).mean(axis=0)
        ).ravel()

In [7]:
marg_pert = {}
for ct in celltypes:
    marg_pert[ct] = {}
    for i, lv in enumerate(levels):
        if ct not in cum_pert or lv not in cum_pert[ct]:
            continue
        if i == 0:
            marg_pert[ct][lv] = cum_pert[ct][lv].copy()
        else:
            prev = levels[i - 1]
            if prev in cum_pert[ct]:
                marg_pert[ct][lv] = cum_pert[ct][lv] - cum_pert[ct][prev]

# Perturbation sharing across levels

In [8]:
rows = []
marg_corrs_list = []
marg_mags_list  = []
cum_corrs_list  = []

for lv in levels:
    cts_ok = [ct for ct in celltypes
              if ct in marg_pert and lv in marg_pert[ct]
              and ct in cum_pert and lv in cum_pert[ct]]
    lv_pairs = list(combinations(cts_ok, 2))

    # Marginal cross-CT correlation
    mc_vals = [pearsonr(marg_pert[a][lv], marg_pert[b][lv])[0]
               for a, b in lv_pairs]
    mc = np.nanmean(mc_vals) if mc_vals else np.nan

    # Marginal magnitude
    mag = np.mean([np.abs(marg_pert[ct][lv]).mean() for ct in cts_ok])

    # Cumulative cross-CT correlation
    cc_vals = [pearsonr(cum_pert[a][lv], cum_pert[b][lv])[0]
               for a, b in lv_pairs]
    cc = np.nanmean(cc_vals) if cc_vals else np.nan

    marg_corrs_list.append(mc)
    marg_mags_list.append(mag)
    cum_corrs_list.append(cc)

    rows.append({
        'level': lv,
        'marginal_cross_ct_r': round(mc, 4),
        'marginal_mean_abs_effect': round(mag, 6),
        'marginal_pct_of_level1': round(mag / marg_mags_list[0] * 100, 1)
                                  if marg_mags_list[0] > 0 else 0,
        'cumulative_cross_ct_r': round(cc, 4),
    })

df_sharing = pd.DataFrame(rows)
df_sharing.to_csv(f'{OUT_DIR}/kang_01_sharing_metrics.csv', index=False)

In [9]:
print(df_sharing.to_string(index=False))

 level  marginal_cross_ct_r  marginal_mean_abs_effect  marginal_pct_of_level1  cumulative_cross_ct_r
     1               0.8841                  0.088662                   100.0                 0.8841
     2               0.2767                  0.023916                    27.0                 0.8569
     3               0.2056                  0.017504                    19.7                 0.8225
     4               0.3119                  0.015415                    17.4                 0.8228
     5               0.1225                  0.018520                    20.9                 0.8068
     6               0.1797                  0.011261                    12.7                 0.8117
     7               0.2443                  0.012141                    13.7                 0.8102
     8               0.2680                  0.009563                    10.8                 0.8101
     9               0.0542                  0.015936                    18.0              

In [10]:
# Plot

fig1, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.plot(levels, marg_corrs_list, '-o', color='#00A087', lw=2, ms=5)
ax.set_xlabel('Latent level')
ax.set_ylabel('Mean pairwise Pearson r')
ax.set_title('Marginal pert effect:\ncross-cell-type similarity')
ax.axhline(0, color='grey', lw=0.5)

ax = axes[1]
ax.plot(levels, marg_mags_list, '-o', color='#00A087', lw=2, ms=5)
ax.set_xlabel('Latent level')
ax.set_ylabel('Mean |marginal effect|')
ax.set_title('Marginal pert effect:\nmagnitude')

ax = axes[2]
ax.plot(levels, cum_corrs_list, '-o', color='#00A087', lw=2, ms=5)
ax.set_xlabel('Latent level')
ax.set_ylabel('Mean pairwise Pearson r')
ax.set_title('Cumulative pert effect:\ncross-cell-type similarity')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
fig1.savefig(f'{OUT_DIR}/kang_pert_sharing_barplot.png', dpi=600, bbox_inches='tight')
fig1.savefig(f'{OUT_DIR}/kang_pert_sharing_barplot.svg', dpi=600, bbox_inches='tight')
plt.close(fig1)

# Shared genes

In [11]:
lv1_mat  = np.stack([marg_pert[ct][levels[0]] for ct in celltypes])
lv1_mean = lv1_mat.mean(axis=0)

shared_idx = np.argsort(-np.abs(lv1_mean))[:30]
shared_rows = []
for rank, idx in enumerate(shared_idx, 1):
    shared_rows.append({
        'rank': rank,
        'gene': gene_names[idx],
        'mean_logFC': round(lv1_mean[idx], 4),
        'is_known_ISG': gene_names[idx] in KNOWN_ISGS,
    })
df_shared = pd.DataFrame(shared_rows)
df_shared.to_csv(f'{OUT_DIR}/kang_shared_genes.csv', index=False)

isgs_found = set(df_shared[df_shared['is_known_ISG']]['gene'])
print(len(isgs_found))

18


In [12]:
# Heatmap

ct_labels = []
for ct in celltypes:
    short = ct.replace(' cells', '').replace(' monocytes', ' mono')
    ct_labels.append(short[:18])

fig2, axes = plt.subplots(1, 1, figsize=(8, max(8, N_SHARED * 0.38)),
                           gridspec_kw={'wspace': 0.5})

shared_data  = lv1_mat[:, shared_idx[:N_SHARED]].T
shared_genes = [gene_names[i] for i in shared_idx[:N_SHARED]]

for ax, data, genes, title in [
    (axes, shared_data, shared_genes,
     f'Shared genes\n(Level 1 marginal effect)')
]:
    vmax = np.abs(data).max()
    if vmax == 0:
        vmax = 1
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im = ax.imshow(data, aspect='auto', cmap='RdBu_r', norm=norm)
    ax.set_xticks(range(len(ct_labels)))
    ax.set_xticklabels(ct_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(genes)))
    ax.set_yticklabels(genes, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.5, label='logFC (marginal)')

plt.suptitle('Kang IFN-β: Shared perturbation genes',
             fontsize=13, fontweight='bold', y=1.01)
fig2.savefig(f'{OUT_DIR}/kang_shared_pert_genes_heatmaps.png', dpi=600, bbox_inches='tight')
fig2.savefig(f'{OUT_DIR}/kang_shared_pert_genes_heatmaps.svg', dpi=600, bbox_inches='tight')
plt.close(fig2)

# GSEAPY

In [13]:
shared_ranking = pd.Series(lv1_mean, index=gene_names).sort_values(ascending=False)
shared_ranking = shared_ranking[~shared_ranking.index.duplicated(keep='first')].dropna()

res_shared = gp.prerank(
        rnk=shared_ranking,
        gene_sets=GSEA_GENESET,
        min_size=15, max_size=500,
        permutation_num=1000, seed=42,
        no_plot=True, verbose=False,
    )
df_gsea_shared = res_shared.res2d.copy()
df_gsea_shared['NES'] = df_gsea_shared['NES'].astype(float)
df_gsea_shared['FDR q-val'] = df_gsea_shared['FDR q-val'].astype(float)
df_gsea_shared.to_csv(f'{OUT_DIR}/kang_gsea_shared.csv', index=False)

In [14]:
df_gsea_shared

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Interferon Gamma Response,0.913583,2.704786,0.0,0.000000,0.0,94/191,2.28%,ISG15;IFIT3;IFIT1;ISG20;LY6E;CXCL10;TNFSF10;MX...
1,prerank,Interferon Alpha Response,0.959819,2.530319,0.0,0.000000,0.0,71/93,1.88%,ISG15;IFIT3;ISG20;LY6E;CXCL10;MX1;IFIT2;IFITM3...
2,prerank,Inflammatory Response,0.694351,2.015161,0.0,0.000000,0.0,41/167,6.20%,LY6E;CXCL10;TNFSF10;IRF7;CXCL11;EIF2AK2;BST2;C...
3,prerank,KRAS Signaling Dn,0.752655,1.882010,0.0,0.000236,0.001,7/66,4.32%,MX1;RSAD2;IFI44L;TGM1;EDN1;SNN;CD80
4,prerank,TNF-alpha Signaling via NF-kB,0.596266,1.762176,0.0,0.003019,0.016,48/192,8.05%,CXCL10;IFIT2;CXCL11;DDX58;SAT1;CCL2;IFIH1;CD69...
5,prerank,Oxidative Phosphorylation,-0.565923,-1.716809,0.0,0.002094,0.002,85/179,18.23%,GPX4;UQCRB;POLR2F;SLC25A11;ECH1;NDUFS8;NDUFS7;...
6,prerank,IL-2/STAT5 Signaling,0.565153,1.666832,0.0,0.009749,0.058,19/173,2.78%,CXCL10;TNFSF10;IFITM3;PLSCR1;GBP4;SOCS1;SELL;R...
7,prerank,Angiogenesis,-0.768193,-1.639167,0.001618,0.005759,0.011,5/20,6.44%,S100A4;VCAN;TIMP1;OLR1;LRPAP1
8,prerank,Myc Targets V1,-0.527209,-1.625318,0.0,0.006632,0.018,92/192,19.11%,RPLP0;HNRNPA1;RPL6;IMPDH2;SLC25A3;HNRNPR;LDHA;...
9,prerank,Apoptosis,0.563393,1.601216,0.004073,0.021295,0.141,29/144,5.35%,ISG20;TNFSF10;IFITM3;SAT1;CD38;CD69;TAP1;DNAJA...
